# RepFR Analyses Runner - Lohnas2025

Run free recall analyses for Lohnas 2025 dataset via papermill.

**Paradigm:** Free Recall
**Status:** Placeholder - awaiting data preparation

In [ ]:
import os
import numpy as np
import papermill as pm

from jaxcmr.helpers import find_project_root, generate_trial_mask, load_data

## Dataset Configuration

In [ ]:
# ============================================================================
# DATASET CONFIGURATION - Lohnas2025
# ============================================================================
# Paradigm: Free Recall (new dataset with mixed/control lists)
# Reference: @lohnas2025temporal
# Status: Data file not yet prepared
# ============================================================================

DATA_NAME = "Lohnas2025"
DATA_FILE = "Lohnas2025.h5"  # TODO: Verify filename when data is ready

# Trial queries - adjust based on actual data structure
QUERIES = {
    "all_lists": "data['list_type'] > 0",      # All experimental lists
    "control": "data['list_type'] == 1",        # Control lists only
    "mixed": "data['list_type'] == 2",          # Mixed lists only  
    "repetition": "data['list_type'] == 4",     # Pure repetition lists
}

# Free recall analyses to run
FREE_RECALL_ANALYSES = ["spc", "crp", "pnr", "repcrp", "backrepcrp", "repneighborcrp"]

In [ ]:
project_root = find_project_root()
templates_dir = os.path.join(project_root, "templates")
rendered_dir = os.path.join(project_root, "projects/repfr/notebooks/rendered")
figures_dir = os.path.join(project_root, "projects/repfr/results/figures/analyses")

os.makedirs(rendered_dir, exist_ok=True)
os.makedirs(figures_dir, exist_ok=True)

data_path = os.path.join(project_root, "data", DATA_FILE)

# Check if data exists
if not os.path.exists(data_path):
    print(f"WARNING: Data file not found at {data_path}")
    print("This orchestrator is a placeholder until data is prepared.")
    DATA_READY = False
else:
    data = load_data(data_path)
    DATA_READY = True
    print(f"Loaded data from {data_path}")

In [ ]:
def list_type_tag(query: str) -> str:
    """Generate a tag based on list types included in query."""
    if not DATA_READY:
        return "placeholder"
    trial_mask = generate_trial_mask(data, query)
    list_types = np.unique(np.array(data["list_type"][trial_mask]).astype(int))
    types_str = "".join(str(t) for t in list_types)
    return f"list_type_{types_str}" if types_str else "list_type_all"

## Analysis Specifications

In [ ]:
trial_query = QUERIES["all_lists"]
control_query = QUERIES["control"]
repcrp_query = QUERIES["repetition"]

analysis_specs = [
    # Basic free recall benchmarks
    {
        "name": "spc",
        "template": "spc.ipynb",
        "parameters": {
            "run_tag": "SPC",
            "data_name": DATA_NAME,
            "data_query": trial_query,
            "output_path": os.path.join(figures_dir, f"spc_{DATA_NAME}.png"),
        },
    },
    {
        "name": "crp",
        "template": "crp.ipynb",
        "parameters": {
            "run_tag": "CRP",
            "data_name": DATA_NAME,
            "data_query": trial_query,
            "output_path": os.path.join(figures_dir, f"crp_{DATA_NAME}.png"),
        },
    },
    {
        "name": "pnr",
        "template": "pnr.ipynb",
        "parameters": {
            "run_tag": "PNR",
            "data_name": DATA_NAME,
            "data_query": trial_query,
            "query_recall_position": 0,
            "output_path": os.path.join(figures_dir, f"pnr_{DATA_NAME}.png"),
        },
    },
    # Repetition-specific analyses
    {
        "name": "repcrp",
        "template": "repcrp.ipynb",
        "parameters": {
            "data_name": DATA_NAME,
            "data_query": repcrp_query,
            "ctrl_query": control_query,
            "min_lag": 4,
            "output_path": os.path.join(figures_dir, f"repcrp_{DATA_NAME}.png"),
        },
    },
    {
        "name": "backrepcrp",
        "template": "backrepcrp.ipynb",
        "parameters": {
            "data_name": DATA_NAME,
            "data_query": repcrp_query,
            "ctrl_query": control_query,
            "min_lag": 4,
            "output_path": os.path.join(figures_dir, f"backrepcrp_{DATA_NAME}.png"),
        },
    },
    {
        "name": "repneighborcrp",
        "template": "repneighborcrp.ipynb",
        "parameters": {
            "data_name": DATA_NAME,
            "data_query": repcrp_query,
            "ctrl_query": control_query,
            "min_lag": 4,
            "output_path": os.path.join(figures_dir, f"repneighborcrp_{DATA_NAME}.png"),
        },
    },
]

## Run Analyses

In [ ]:
if not DATA_READY:
    print("Skipping analyses - data not yet available.")
else:
    for spec in analysis_specs:
        template_path = os.path.join(templates_dir, spec["template"])
        query = spec["parameters"].get("data_query")
        tag = list_type_tag(query) if query else "list_type_all"
        rendered_name = f"{spec['name']}_{DATA_NAME}_{tag}.ipynb"
        output_path = os.path.join(rendered_dir, rendered_name)
        
        print(f"Running {spec['name']} -> {output_path}")
        pm.execute_notebook(
            template_path,
            output_path,
            parameters=spec["parameters"],
        )